# SDAHU: raw windows, full-train batched XGBoost — fixed trajectory slicing

Этот ноутбук:
- загружает `train/val/test` из CSV с `MultiIndex`
- **режет окна строго внутри траектории**, где траектория определяется **первым уровнем MultiIndex**
- корректно выравнивает `target` по индексу **после сортировки**
- проверяет, что внутри каждой траектории таргет константен
- обучает `XGBoost` на полном train батчами
- делает streaming inference для `val/test`

Важно:
- здесь предполагается, что одна траектория соответствует одному классу
- для индекса вида `(class, date)` траектория = первый уровень индекса, время = второй уровень индекса


In [1]:
import gc
import time
import warnings
from os.path import join

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sklearn.metrics import accuracy_score, f1_score, log_loss, classification_report

import xgboost as xgb


In [2]:

folder = r'/home/ilya_treyvish/projects/lbnl_fdd/data/processed/SDAHU'

train_df = pd.read_csv(join(folder, 'train_df.csv'), index_col=[0, 1], parse_dates=[1])
train_target = pd.read_csv(join(folder, 'train_target.csv'), index_col=[0, 1], parse_dates=[1])

val_df = pd.read_csv(join(folder, 'val_df.csv'), index_col=[0, 1], parse_dates=[1])
val_target = pd.read_csv(join(folder, 'val_target.csv'), index_col=[0, 1], parse_dates=[1])

test_df = pd.read_csv(join(folder, 'test_df.csv'), index_col=[0, 1], parse_dates=[1])
test_target = pd.read_csv(join(folder, 'test_target.csv'), index_col=[0, 1], parse_dates=[1])

print("train_df:", train_df.shape)
print("train_target:", train_target.shape)
print("val_df:", val_df.shape)
print("val_target:", val_target.shape)
print("test_df:", test_df.shape)
print("test_target:", test_target.shape)


train_df: (4448700, 25)
train_target: (4448700, 1)
val_df: (1317600, 25)
val_target: (1317600, 1)
test_df: (1899361, 25)
test_target: (1899361, 1)


In [3]:

train_df.head()


CHWC_VLV_DM  ZONE_TEMP_4    SA_CFM  \
fault_type Datetime                                                  
0          2018-01-01 01:00:00          0.0    66.761345 -0.943331   
           2018-01-01 01:01:00          0.0    66.761290 -0.943167   
           2018-01-01 01:02:00          0.0    66.761185 -0.943027   
           2018-01-01 01:03:00          0.0    66.761020 -0.942867   
           2018-01-01 01:04:00          0.0    66.760740 -0.942719   

                                      SF_WAT    MA_TEMP  SYS_CTL  OA_DMPR_DM  \
fault_type Datetime                                                            
0          2018-01-01 01:00:00 -2.791519e-13  66.374680      0.0         0.0   
           2018-01-01 01:01:00 -2.790612e-13  66.374680      0.0         0.0   
           2018-01-01 01:02:00 -2.789696e-13  66.374680      0.0         0.0   
           2018-01-01 01:03:00 -2.788807e-13  66.374626      0.0         0.0   
           2018-01-01 01:04:00 -2.787895e-13  66.374626      0.0         0.0   

                                ZONE_TEMP_2   RA_TEMP      CHWC_VLV  ...  \
fault_type Datetime                                                  ...   
0          2018-01-01 01:00:00    66.767230  68.34175  2.635303e-21  ...   
           2018-01-01 01:01:00    66.766680  68.34786  2.479578e-21  ...   
           2018-01-01 01:02:00    66.766014  68.35378  2.380361e-21  ...   
           2018-01-01 01:03:00    66.765305  68.35948 -1.274259e-21  ...   
           2018-01-01 01:04:00    66.764530  68.36498  2.749987e-21  ...   

                                RA_DMPR_DM    SA_TEMP  RF_CS  RA_DMPR  \
fault_type Datetime                                                     
0          2018-01-01 01:00:00         1.0  66.374680    0.0      1.0   
           2018-01-01 01:01:00         1.0  66.374680    0.0      1.0   
           2018-01-01 01:02:00         1.0  66.374680    0.0      1.0   
           2018-01-01 01:03:00         1.0  66.374626    0.0      1.0   
           2018-01-01 01:04:00         1.0  66.374626    0.0      1.0   

                                  RA_CFM  RF_SPD_DM        RF_WAT  \
fault_type Datetime                                                 
0          2018-01-01 01:00:00  0.853194        0.0 -2.617164e-13   
           2018-01-01 01:01:00  0.853057        0.0 -2.616267e-13   
           2018-01-01 01:02:00  0.852918        0.0 -2.615462e-13   
           2018-01-01 01:03:00  0.852779        0.0 -2.614546e-13   
           2018-01-01 01:04:00  0.852641        0.0 -2.613691e-13   

                                ZONE_TEMP_1  ZONE_TEMP_3  SF_SPD_DM  
fault_type Datetime                                                  
0          2018-01-01 01:00:00     73.94652    67.027100        0.0  
           2018-01-01 01:01:00     73.97777    67.025894        0.0  
           2018-01-01 01:02:00     74.00836    67.024580        0.0  
           2018-01-01 01:03:00     74.03825    67.023150        0.0  
           2018-01-01 01:04:00     74.06741    67.021670        0.0  

[5 rows x 25 columns]

In [4]:

train_target.head()


target
fault_type Datetime                   
0          2018-01-01 01:00:00       0
           2018-01-01 01:01:00       0
           2018-01-01 01:02:00       0
           2018-01-01 01:03:00       0
           2018-01-01 01:04:00       0

In [5]:

def ensure_target_series(target: pd.DataFrame | pd.Series | np.ndarray) -> pd.Series:
    """
    Приводит target к Series.
    Поддерживает:
    - Series
    - DataFrame с 1 колонкой
    - ndarray shape (n,) или (n, 1)
    """
    if isinstance(target, pd.Series):
        return target

    if isinstance(target, pd.DataFrame):
        if target.shape[1] != 1:
            raise ValueError(
                f"У target DataFrame {target.shape[1]} столбцов; ожидался 1 столбец."
            )
        return target.iloc[:, 0]

    arr = np.asarray(target)
    if arr.ndim == 2:
        if arr.shape[1] != 1:
            raise ValueError(f"target имеет shape={arr.shape}; ожидался shape (n,) или (n, 1).")
        arr = arr.reshape(-1)
    elif arr.ndim != 1:
        raise ValueError(f"target имеет ndim={arr.ndim}; ожидался 1D или 2D c одним столбцом.")

    return pd.Series(arr)


def infer_index_levels(df: pd.DataFrame) -> tuple[str, str]:
    """
    Для MultiIndex вида (trajectory_id, time) возвращает имена уровней.
    По умолчанию считаем:
    - первый уровень = идентификатор траектории / класса
    - второй уровень = время
    """
    if not isinstance(df.index, pd.MultiIndex):
        raise ValueError("Ожидался MultiIndex у df.")

    if df.index.nlevels < 2:
        raise ValueError(
            f"Ожидался MultiIndex минимум с 2 уровнями, получено nlevels={df.index.nlevels}"
        )

    level_names = list(df.index.names)
    trajectory_level = level_names[0] if level_names[0] is not None else 0
    time_level = level_names[1] if level_names[1] is not None else 1
    return trajectory_level, time_level


def align_target_to_df_index(df: pd.DataFrame, target) -> pd.Series:
    """
    Безопасно выравнивает target к df по значению индекса, а не по текущему порядку строк.
    Это исправляет сценарий, когда df был отсортирован, а target — ещё нет.
    """
    y = ensure_target_series(target)

    if len(y) != len(df):
        raise ValueError(
            f"Длины df и target не совпадают: len(df)={len(df)}, len(target)={len(y)}"
        )

    if isinstance(y.index, pd.MultiIndex):
        missing_in_target = df.index.difference(y.index)
        if len(missing_in_target) > 0:
            raise ValueError(
                f"У target отсутствуют {len(missing_in_target)} индексов из df. "
                f"Пример: {list(missing_in_target[:3])}"
            )
        y_aligned = y.loc[df.index]
        return pd.Series(np.asarray(y_aligned).reshape(-1), index=df.index, name=y.name)

    return pd.Series(np.asarray(y).reshape(-1), index=df.index, name=getattr(y, "name", None))


def make_label_encoder(classes_: np.ndarray) -> tuple[dict, np.ndarray]:
    """
    Возвращает:
    - class_to_idx: исходный класс -> индекс 0..K-1
    - idx_to_class: np.ndarray, где idx_to_class[i] = исходный класс
    """
    idx_to_class = np.asarray(sorted(pd.unique(np.asarray(classes_).reshape(-1))))
    class_to_idx = {cls: i for i, cls in enumerate(idx_to_class.tolist())}
    return class_to_idx, idx_to_class


def prepare_grouped_arrays(
    df: pd.DataFrame,
    target,
    class_to_idx: dict | None = None,
) -> tuple[list, list, dict]:
    """
    Возвращает список групп (траекторий), где каждая группа содержит:
    - X: np.ndarray shape (T, F)
    - y: np.ndarray shape (T,) с исходным классом
    - y_encoded: np.ndarray shape (T,) с индексом класса 0..K-1
    - trajectory_id: значение первого уровня MultiIndex
    - index: исходный MultiIndex группы

    Предполагается, что одна траектория полностью принадлежит одному классу.
    """
    trajectory_level, time_level = infer_index_levels(df)

    x = df.copy().sort_index(level=[trajectory_level, time_level])
    y_aligned = align_target_to_df_index(x, target)

    num_cols = [c for c in x.columns if pd.api.types.is_numeric_dtype(x[c])]
    if not num_cols:
        raise ValueError("В df не найдено числовых признаков для XGBoost.")

    feature_names = num_cols
    groups = []

    for trajectory_id, g in x.groupby(level=trajectory_level, sort=False):
        g_idx = g.index
        yg = y_aligned.loc[g_idx].to_numpy().reshape(-1)

        unique_classes = pd.unique(yg)
        if len(unique_classes) != 1:
            raise ValueError(
                f"Внутри траектории {trajectory_id!r} найдено несколько классов: {unique_classes[:10]}. "
                "Ожидалось, что одна траектория целиком принадлежит одному классу."
            )

        Xg = g[num_cols].to_numpy(dtype=np.float32)
        class_value = unique_classes[0]
        class_idx = None if class_to_idx is None else int(class_to_idx[class_value])

        groups.append({
            "trajectory_id": trajectory_id,
            "class_value": class_value,
            "class_idx": class_idx,
            "X": Xg,
            "y": np.full(len(Xg), class_value),
            "y_encoded": None if class_idx is None else np.full(len(Xg), class_idx, dtype=np.int32),
            "index": g_idx,
        })

    meta = {
        "trajectory_level": trajectory_level,
        "time_level": time_level,
        "n_groups": len(groups),
    }
    return groups, feature_names, meta


In [6]:

train_df_sorted = train_df.copy().sort_index(level=[train_df.index.names[0], train_df.index.names[1]])
train_target_s = align_target_to_df_index(train_df_sorted, train_target)
classes_ = np.sort(pd.unique(train_target_s.to_numpy().reshape(-1)))
class_to_idx, idx_to_class = make_label_encoder(classes_)

train_groups, feature_names, train_meta = prepare_grouped_arrays(train_df, train_target, class_to_idx=class_to_idx)
val_groups, _, val_meta = prepare_grouped_arrays(val_df, val_target, class_to_idx=class_to_idx)
test_groups, _, test_meta = prepare_grouped_arrays(test_df, test_target, class_to_idx=class_to_idx)

n_features = len(feature_names)

print("trajectory level:", train_meta["trajectory_level"])
print("time level:", train_meta["time_level"])
print("classes_:", classes_)
print("class_to_idx:", class_to_idx)
print("labels_are_0_to_k_minus_1:", np.array_equal(classes_, np.arange(len(classes_))))
print("n_features:", n_features)
print("n_train_groups:", len(train_groups))
print("example trajectory_id:", train_groups[0]["trajectory_id"])
print("example class_value:", train_groups[0]["class_value"])
print("example class_idx:", train_groups[0]["class_idx"])
print("example X shape:", train_groups[0]["X"].shape)
print("example y unique:", np.unique(train_groups[0]["y"]))
print("example y_encoded unique:", np.unique(train_groups[0]["y_encoded"]))


trajectory level: fault_type
time level: Datetime
classes_: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14]
class_to_idx: {0: 0, 1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6, 7: 7, 8: 8, 9: 9, 10: 10, 11: 11, 12: 12, 13: 13, 14: 14}
labels_are_0_to_k_minus_1: True
n_features: 25
n_train_groups: 15
example trajectory_id: 0
example class_value: 0
example class_idx: 0
example X shape: (305220, 25)
example y unique: [0]
example y_encoded unique: [0]


In [7]:

# sanity checks: одна траектория = один класс, индекс отсортирован внутри траектории
for split_name, groups, meta in [
    ("train", train_groups, train_meta),
    ("val", val_groups, val_meta),
    ("test", test_groups, test_meta),
]:
    print(f"--- {split_name} ---")
    print("n_groups:", len(groups))
    print("trajectory level:", meta["trajectory_level"], "| time level:", meta["time_level"])
    for g in groups[:3]:
        uniq = np.unique(g["y"])
        print(
            "trajectory_id=", g["trajectory_id"],
            "| class=", g["class_value"],
            "| len=", len(g["X"]),
            "| unique_y=", uniq
        )


--- train ---
n_groups: 15
trajectory level: fault_type | time level: Datetime
trajectory_id= 0 | class= 0 | len= 305220 | unique_y= [0]
trajectory_id= 1 | class= 1 | len= 305220 | unique_y= [1]
trajectory_id= 2 | class= 2 | len= 305220 | unique_y= [2]
--- val ---
n_groups: 15
trajectory level: fault_type | time level: Datetime
trajectory_id= 0 | class= 0 | len= 87840 | unique_y= [0]
trajectory_id= 1 | class= 1 | len= 87840 | unique_y= [1]
trajectory_id= 2 | class= 2 | len= 87840 | unique_y= [2]
--- test ---
n_groups: 15
trajectory level: fault_type | time level: Datetime
trajectory_id= 0 | class= 0 | len= 132480 | unique_y= [0]
trajectory_id= 1 | class= 1 | len= 132480 | unique_y= [1]
trajectory_id= 2 | class= 2 | len= 132480 | unique_y= [2]


## Диагностика нарезки окон

Ниже — ячейки для визуальной и количественной проверки, что:

- одна траектория соответствует одному классу;
- окна режутся **только внутри** траектории;
- начало и конец окна попадают в корректные даты;
- число окон по траекториям считается ожидаемо: `len(traj) - window + 1`.


In [8]:

def summarize_groups(groups, split_name: str, max_rows: int = 10) -> pd.DataFrame:
    rows = []
    for g in groups:
        idx = g["index"]
        times = idx.get_level_values(-1)
        rows.append({
            "split": split_name,
            "trajectory_id": g["trajectory_id"],
            "class_value": g["class_value"],
            "n_points": len(g["X"]),
            "n_windows_w1": max(0, len(g["X"]) - 1 + 1),
            "n_windows_w10": max(0, len(g["X"]) - 10 + 1),
            "n_windows_w100": max(0, len(g["X"]) - 100 + 1),
            "start_time": times.min(),
            "end_time": times.max(),
            "is_time_monotonic": pd.Index(times).is_monotonic_increasing,
        })
    out = pd.DataFrame(rows).sort_values(["split", "trajectory_id"]).reset_index(drop=True)
    return out.head(max_rows)

display(summarize_groups(train_groups, "train", max_rows=20))
display(summarize_groups(val_groups, "val", max_rows=20))
display(summarize_groups(test_groups, "test", max_rows=20))


,split,trajectory_id,class_value,n_points,n_windows_w1,n_windows_w10,n_windows_w100,start_time,end_time,is_time_monotonic
0,train,0,0,305220,305220,305211,305121,2018-01-01 01:00:00,2018-07-31 23:59:00,True
1,train,1,1,305220,305220,305211,305121,2018-01-01 01:00:00,2018-07-31 23:59:00,True
2,train,2,2,305220,305220,305211,305121,2018-01-01 01:00:00,2018-07-31 23:59:00,True
3,train,3,3,305220,305220,305211,305121,2018-01-01 01:00:00,2018-07-31 23:59:00,True
4,train,4,4,305220,305220,305211,305121,2018-01-01 01:00:00,2018-07-31 23:59:00,True
5,train,5,5,305220,305220,305211,305121,2018-01-01 01:00:00,2018-07-31 23:59:00,True
6,train,6,6,305220,305220,305211,305121,2018-01-01 01:00:00,2018-07-31 23:59:00,True
7,train,7,7,305220,305220,305211,305121,2018-01-01 01:00:00,2018-07-31 23:59:00,True
8,train,8,8,305220,305220,305211,305121,2018-01-01 01:00:00,2018-07-31 23:59:00,True
9,train,9,9,305220,305220,305211,305121,2018-01-01 01:00:00,2018-07-31 23:59:00,True


,split,trajectory_id,class_value,n_points,n_windows_w1,n_windows_w10,n_windows_w100,start_time,end_time,is_time_monotonic
0,val,0,0,87840,87840,87831,87741,2018-08-01,2018-09-30 23:59:00,True
1,val,1,1,87840,87840,87831,87741,2018-08-01,2018-09-30 23:59:00,True
2,val,2,2,87840,87840,87831,87741,2018-08-01,2018-09-30 23:59:00,True
3,val,3,3,87840,87840,87831,87741,2018-08-01,2018-09-30 23:59:00,True
4,val,4,4,87840,87840,87831,87741,2018-08-01,2018-09-30 23:59:00,True
5,val,5,5,87840,87840,87831,87741,2018-08-01,2018-09-30 23:59:00,True
6,val,6,6,87840,87840,87831,87741,2018-08-01,2018-09-30 23:59:00,True
7,val,7,7,87840,87840,87831,87741,2018-08-01,2018-09-30 23:59:00,True
8,val,8,8,87840,87840,87831,87741,2018-08-01,2018-09-30 23:59:00,True
9,val,9,9,87840,87840,87831,87741,2018-08-01,2018-09-30 23:59:00,True


,split,trajectory_id,class_value,n_points,n_windows_w1,n_windows_w10,n_windows_w100,start_time,end_time,is_time_monotonic
0,test,0,0,132480,132480,132471,132381,2018-10-01,2018-12-31 23:59:00,True
1,test,1,1,132480,132480,132471,132381,2018-10-01,2018-12-31 23:59:00,True
2,test,2,2,132480,132480,132471,132381,2018-10-01,2018-12-31 23:59:00,True
3,test,3,3,132480,132480,132471,132381,2018-10-01,2018-12-31 23:59:00,True
4,test,4,4,132480,132480,132471,132381,2018-10-01,2018-12-31 23:59:00,True
5,test,5,5,132480,132480,132471,132381,2018-10-01,2018-12-31 23:59:00,True
6,test,6,6,132480,132480,132471,132381,2018-10-01,2018-12-31 23:59:00,True
7,test,7,7,132480,132480,132471,132381,2018-10-01,2018-12-31 23:59:00,True
8,test,8,8,132480,132480,132471,132381,2018-10-01,2018-12-31 23:59:00,True
9,test,9,9,132480,132480,132471,132381,2018-10-01,2018-12-31 23:59:00,True


In [9]:

def make_window_offsets(window: int, sparse_step: int | None = None) -> np.ndarray:
    """
    Возвращает смещения внутри окна, которые нужно взять.
    - sparse_step=None -> берём все точки окна
    - sparse_step=10, window=100 -> берём позиции 0, 10, ..., 90
    """
    if sparse_step is None:
        return np.arange(window, dtype=np.int64)

    offsets = np.arange(0, window, sparse_step, dtype=np.int64)
    if len(offsets) == 0:
        raise ValueError(
            f"Для window={window} и sparse_step={sparse_step} не получилось построить offsets"
        )
    return offsets


def make_window_offset(window: int, sparse_step: int | None = None) -> np.ndarray:
    """Backward-compatible alias for make_window_offsets."""
    return make_window_offsets(window=window, sparse_step=sparse_step)


def valid_window_endpoints(group_length: int, window: int) -> np.ndarray:
    if group_length < window:
        return np.array([], dtype=np.int64)
    return np.arange(window - 1, group_length, dtype=np.int64)


def flatten_window(Xg: np.ndarray, end: int, window: int, offsets: np.ndarray) -> np.ndarray:
    start = end - window + 1
    return Xg[start + offsets].reshape(-1)


def count_total_windows(groups, window: int) -> int:
    total = 0
    for g in groups:
        total += max(0, len(g["X"]) - window + 1)
    return total


def build_sampled_train_windows(
    groups,
    window: int,
    total_samples: int,
    random_state: int = 42,
    sparse_step: int | None = None,
    save_selection_path: str | None = None,
):
    """
    Строит raw-window train выборку:
    - окно flatten в вектор длины len(offsets) * n_features
    - сэмпл стратифицирован по классам
    - генерируются только выбранные окна
    - при необходимости сохраняется точный список использованных окон
    """
    rng = np.random.default_rng(random_state)
    offsets = make_window_offsets(window=window, sparse_step=sparse_step)

    class_to_candidates = {}
    for group_idx, g in enumerate(groups):
        endpoints = valid_window_endpoints(len(g["X"]), window)
        if len(endpoints) == 0:
            continue
        for end in endpoints:
            cls = g["class_value"]
            class_to_candidates.setdefault(cls, []).append((group_idx, int(end)))

    classes = sorted(class_to_candidates.keys())
    if not classes:
        raise ValueError(f"Не найдено ни одного окна для window={window}")

    base_quota = total_samples // len(classes)
    remainder = total_samples % len(classes)

    selected = []
    shortages = 0

    for i, cls in enumerate(classes):
        candidates = class_to_candidates[cls]
        need = base_quota + (1 if i < remainder else 0)
        take = min(need, len(candidates))
        if take > 0:
            idx = rng.choice(len(candidates), size=take, replace=False)
            selected.extend([candidates[j] for j in idx])
        shortages += max(0, need - take)

    if shortages > 0:
        remaining = []
        selected_set = set(selected)
        for cls in classes:
            for item in class_to_candidates[cls]:
                if item not in selected_set:
                    remaining.append(item)

        if remaining:
            take = min(shortages, len(remaining))
            idx = rng.choice(len(remaining), size=take, replace=False)
            selected.extend([remaining[j] for j in idx])

    rng.shuffle(selected)

    n_samples = len(selected)
    if n_samples == 0:
        raise ValueError(f"После сэмплирования не осталось окон для window={window}")

    flat_dim = len(offsets) * groups[0]["X"].shape[1]
    X = np.empty((n_samples, flat_dim), dtype=np.float32)
    y = np.empty(n_samples, dtype=np.int64)
    records = []

    for i, (group_idx, end) in enumerate(selected):
        g = groups[group_idx]
        X[i] = flatten_window(g["X"], end=end, window=window, offsets=offsets)
        y[i] = np.asarray(g["class_idx"]).item() if g["class_idx"] is not None else np.asarray(g["class_value"]).item()

        records.append({
            "sample_id": i,
            "group_idx": group_idx,
            "trajectory_id": g["trajectory_id"],
            "class_value": g["class_value"],
            "class_idx": g["class_idx"],
            "window": window,
            "sparse_step": -1 if sparse_step is None else sparse_step,
            "endpoint_pos": int(end),
            "start_pos": int(end - window + 1),
            "label": int(y[i]),
            "endpoint_time": str(g["index"][end][1]),
            "offsets": ','.join(map(str, offsets.tolist())),
        })

    selection_df = pd.DataFrame(records)

    if save_selection_path is not None:
        os.makedirs(os.path.dirname(save_selection_path), exist_ok=True)
        selection_df.to_json(save_selection_path, orient="records", lines=True)
        print(f"Saved train window selection to: {save_selection_path}")

    return X, y, selection_df, offsets


def stream_windows(
    groups,
    window: int,
    batch_size: int = 4096,
    sparse_step: int | None = None,
    label_mode: str = "raw",
):
    """
    Простой последовательный генератор батчей окон.
    Ничего не материализует целиком.
    Yield: X_batch, y_batch
    """
    offsets = make_window_offsets(window=window, sparse_step=sparse_step)
    X_parts = []
    y_parts = []

    for g in groups:
        Xg = g["X"]

        if len(Xg) < window:
            continue

        label = g["class_value"] if label_mode == "raw" else g["class_idx"]
        if label_mode == "encoded" and label is None:
            raise ValueError("Для label_mode='encoded' у групп должен быть заполнен class_idx.")

        for end in range(window - 1, len(Xg)):
            X_parts.append(flatten_window(Xg, end=end, window=window, offsets=offsets))
            y_parts.append(int(label))

            if len(X_parts) >= batch_size:
                yield np.asarray(X_parts, dtype=np.float32), np.asarray(y_parts, dtype=np.int64)
                X_parts = []
                y_parts = []

    if X_parts:
        yield np.asarray(X_parts, dtype=np.float32), np.asarray(y_parts, dtype=np.int64)


def stream_windows_mixed(
    groups,
    window: int,
    batch_size: int = 4096,
    sparse_step: int | None = None,
    label_mode: str = "encoded",
    random_state: int = 42,
):
    """
    Смешанный генератор train-батчей:
    - окна не идут блоками по одной траектории/классу
    - на каждом проходе берём по одному следующему окну из разных групп
    - порядок групп в каждом цикле перемешивается

    Это важно для случая "одна траектория = один класс", чтобы booster
    не дообучался длинными сериями окон одного и того же класса.
    """
    rng = np.random.default_rng(random_state)
    offsets = make_window_offsets(window=window, sparse_step=sparse_step)

    group_infos = []
    for g in groups:
        endpoints = valid_window_endpoints(len(g["X"]), window)
        if len(endpoints) == 0:
            continue

        label = g["class_value"] if label_mode == "raw" else g["class_idx"]
        if label_mode == "encoded" and label is None:
            raise ValueError("Для label_mode='encoded' у групп должен быть заполнен class_idx.")

        group_infos.append({
            "X": g["X"],
            "trajectory_id": g["trajectory_id"],
            "class_value": g["class_value"],
            "class_idx": g["class_idx"],
            "endpoints": endpoints,
            "next_pos": 0,
            "label": int(label),
        })

    if not group_infos:
        return

    X_parts = []
    y_parts = []

    while True:
        active = [i for i, info in enumerate(group_infos) if info["next_pos"] < len(info["endpoints"])]
        if not active:
            break

        rng.shuffle(active)

        for gi in active:
            info = group_infos[gi]
            end = int(info["endpoints"][info["next_pos"]])
            info["next_pos"] += 1

            X_parts.append(flatten_window(info["X"], end=end, window=window, offsets=offsets))
            y_parts.append(info["label"])

            if len(X_parts) >= batch_size:
                yield np.asarray(X_parts, dtype=np.float32), np.asarray(y_parts, dtype=np.int64)
                X_parts = []
                y_parts = []

    if X_parts:
        yield np.asarray(X_parts, dtype=np.float32), np.asarray(y_parts, dtype=np.int64)


def inspect_train_batch_mixing(
    groups,
    window: int,
    batch_size: int = 50_000,
    sparse_step: int | None = None,
    random_state: int = 42,
    n_batches: int = 5,
    idx_to_class: np.ndarray | None = None,
):
    rows = []
    for batch_idx, (_, yb) in enumerate(
        stream_windows_mixed(
            groups,
            window=window,
            batch_size=batch_size,
            sparse_step=sparse_step,
            label_mode="encoded",
            random_state=random_state,
        )
    ):
        uniq, cnt = np.unique(yb, return_counts=True)
        row = {
            "batch_idx": batch_idx,
            "batch_size": int(len(yb)),
            "n_classes_in_batch": int(len(uniq)),
            "max_class_share": float(cnt.max() / cnt.sum()),
        }
        for cls_idx, c in zip(uniq.tolist(), cnt.tolist()):
            key = f"class_{cls_idx}"
            if idx_to_class is not None:
                key = f"class_{idx_to_class[cls_idx]}"
            row[key] = int(c)
        rows.append(row)
        if batch_idx + 1 >= n_batches:
            break
    return pd.DataFrame(rows)


def fit_xgb_batched(
    train_groups,
    window: int,
    classes_: np.ndarray,
    batch_size: int = 50_000,
    sparse_step: int | None = None,
    random_state: int = 42,
    rounds_per_batch: int = 25,
    xgb_params: dict | None = None,
):
    """
    Обучение XGBoost на полном train батчами.

    Важно:
    - метки обязательно перекодируются в 0..K-1
    - train-батчи смешиваются между траекториями/классами
    - это всё ещё не sklearn partial_fit, а последовательное доращивание booster
      через xgb_model=prev_booster, но уже без перекоса из-за длинных одно-классовых блоков
    """
    if xgb_params is None:
        xgb_params = {
            "objective": "multi:softprob",
            "num_class": int(len(classes_)),
            "max_depth": 6,
            "eta": 0.1,
            "subsample": 0.8,
            "colsample_bytree": 0.8,
            "tree_method": "hist",
            "eval_metric": "mlogloss",
            "seed": random_state,
            "nthread": -1,
        }

    booster = None
    total_rows = 0
    n_batches = 0
    batch_class_stats = []

    for Xb, yb in stream_windows_mixed(
        train_groups,
        window=window,
        batch_size=batch_size,
        sparse_step=sparse_step,
        label_mode="encoded",
        random_state=random_state,
    ):
        uniq, cnt = np.unique(yb, return_counts=True)
        batch_class_stats.append({
            "batch_idx": n_batches,
            "batch_size": int(len(yb)),
            "n_classes_in_batch": int(len(uniq)),
            "max_class_share": float(cnt.max() / cnt.sum()),
        })

        dtrain = xgb.DMatrix(Xb, label=yb, feature_names=None)
        booster = xgb.train(
            params=xgb_params,
            dtrain=dtrain,
            num_boost_round=rounds_per_batch,
            xgb_model=booster,
            verbose_eval=False,
        )
        total_rows += len(yb)
        n_batches += 1

        del Xb, yb, dtrain
        gc.collect()

    if booster is None:
        raise ValueError(f"Не удалось построить ни одного train-батча для window={window}")

    return XGBStreamingWrapper(booster=booster, classes_=classes_), {
        "train_rows_seen": total_rows,
        "train_batches": n_batches,
        "rounds_per_batch": rounds_per_batch,
        "total_boost_rounds": n_batches * rounds_per_batch,
        "batch_class_stats": pd.DataFrame(batch_class_stats),
    }


class XGBStreamingWrapper:
    def __init__(self, booster, classes_):
        self.booster = booster
        self.classes_ = np.asarray(classes_)

    def predict_proba(self, X):
        d = xgb.DMatrix(X, feature_names=None)
        proba = self.booster.predict(d)
        if proba.ndim == 1:
            proba = np.column_stack([1.0 - proba, proba])
        return proba

    def predict(self, X):
        proba = self.predict_proba(X)
        return self.classes_[np.argmax(proba, axis=1)]


def evaluate_streaming(
    model,
    groups,
    window: int,
    classes_: np.ndarray,
    batch_size: int = 4096,
    sparse_step: int | None = None,
):
    """
    Streaming evaluation для val/test.
    Поддерживает accuracy, macro_f1, weighted_f1, logloss.
    """
    y_true_all = []
    y_pred_all = []
    y_proba_all = []

    has_proba = hasattr(model, "predict_proba")

    for Xb, yb in stream_windows(
        groups,
        window=window,
        batch_size=batch_size,
        sparse_step=sparse_step,
        label_mode="raw",
    ):
        pred = model.predict(Xb)
        y_true_all.append(yb)
        y_pred_all.append(pred)

        if has_proba:
            proba = model.predict_proba(Xb)
            y_proba_all.append(proba)

    y_true = np.concatenate(y_true_all)
    y_pred = np.concatenate(y_pred_all)

    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }

    if has_proba and len(y_proba_all) > 0:
        y_proba = np.concatenate(y_proba_all)
        metrics["logloss"] = log_loss(y_true, y_proba, labels=classes_)
    else:
        metrics["logloss"] = np.nan

    return metrics, y_true, y_pred


In [10]:

def inspect_windows_for_group(
    groups,
    window: int,
    sparse_step: int | None = None,
    group_pos: int = 0,
    n_examples: int = 3,
) -> pd.DataFrame:
    offsets = make_window_offsets(window=window, sparse_step=sparse_step)
    g = groups[group_pos]
    endpoints = valid_window_endpoints(len(g["X"]), window)

    rows = []
    for end in endpoints[:n_examples]:
        start = end - window + 1
        idx_slice = g["index"][start:end + 1]
        sampled_idx = g["index"][start + offsets]
        full_times = idx_slice.get_level_values(-1)
        sampled_times = sampled_idx.get_level_values(-1)

        rows.append({
            "trajectory_id": g["trajectory_id"],
            "class_value": g["class_value"],
            "window": window,
            "sparse_step": sparse_step,
            "start_pos": int(start),
            "end_pos": int(end),
            "n_points_full_window": len(idx_slice),
            "n_points_used": len(sampled_idx),
            "full_start_time": full_times.min(),
            "full_end_time": full_times.max(),
            "sampled_start_time": sampled_times.min(),
            "sampled_end_time": sampled_times.max(),
            "sampled_points_preview": list(sampled_idx[:min(5, len(sampled_idx))]),
        })

    return pd.DataFrame(rows)

display(inspect_windows_for_group(train_groups, window=10, sparse_step=None, group_pos=0, n_examples=5))
display(inspect_windows_for_group(train_groups, window=100, sparse_step=10, group_pos=0, n_examples=5))


,trajectory_id,class_value,window,sparse_step,start_pos,end_pos,n_points_full_window,n_points_used,full_start_time,full_end_time,sampled_start_time,sampled_end_time,sampled_points_preview
0,0,0,10,None,0,9,10,10,2018-01-01 01:00:00,2018-01-01 01:09:00,2018-01-01 01:00:00,2018-01-01 01:09:00,"[(0, 2018-01-01 01:00:00), (0, 2018-01-01 01:0..."
1,0,0,10,None,1,10,10,10,2018-01-01 01:01:00,2018-01-01 01:10:00,2018-01-01 01:01:00,2018-01-01 01:10:00,"[(0, 2018-01-01 01:01:00), (0, 2018-01-01 01:0..."
2,0,0,10,None,2,11,10,10,2018-01-01 01:02:00,2018-01-01 01:11:00,2018-01-01 01:02:00,2018-01-01 01:11:00,"[(0, 2018-01-01 01:02:00), (0, 2018-01-01 01:0..."
3,0,0,10,None,3,12,10,10,2018-01-01 01:03:00,2018-01-01 01:12:00,2018-01-01 01:03:00,2018-01-01 01:12:00,"[(0, 2018-01-01 01:03:00), (0, 2018-01-01 01:0..."
4,0,0,10,None,4,13,10,10,2018-01-01 01:04:00,2018-01-01 01:13:00,2018-01-01 01:04:00,2018-01-01 01:13:00,"[(0, 2018-01-01 01:04:00), (0, 2018-01-01 01:0..."


,trajectory_id,class_value,window,sparse_step,start_pos,end_pos,n_points_full_window,n_points_used,full_start_time,full_end_time,sampled_start_time,sampled_end_time,sampled_points_preview
0,0,0,100,10,0,99,100,10,2018-01-01 01:00:00,2018-01-01 02:39:00,2018-01-01 01:00:00,2018-01-01 02:30:00,"[(0, 2018-01-01 01:00:00), (0, 2018-01-01 01:1..."
1,0,0,100,10,1,100,100,10,2018-01-01 01:01:00,2018-01-01 02:40:00,2018-01-01 01:01:00,2018-01-01 02:31:00,"[(0, 2018-01-01 01:01:00), (0, 2018-01-01 01:1..."
2,0,0,100,10,2,101,100,10,2018-01-01 01:02:00,2018-01-01 02:41:00,2018-01-01 01:02:00,2018-01-01 02:32:00,"[(0, 2018-01-01 01:02:00), (0, 2018-01-01 01:1..."
3,0,0,100,10,3,102,100,10,2018-01-01 01:03:00,2018-01-01 02:42:00,2018-01-01 01:03:00,2018-01-01 02:33:00,"[(0, 2018-01-01 01:03:00), (0, 2018-01-01 01:1..."
4,0,0,100,10,4,103,100,10,2018-01-01 01:04:00,2018-01-01 02:43:00,2018-01-01 01:04:00,2018-01-01 02:34:00,"[(0, 2018-01-01 01:04:00), (0, 2018-01-01 01:1..."


In [11]:

def validate_window_slicing(groups, windows=(1, 10, 100), sparse_step_map=None) -> pd.DataFrame:
    if sparse_step_map is None:
        sparse_step_map = {1: None, 10: None, 100: None}

    rows = []
    for window in windows:
        sparse_step = sparse_step_map.get(window, None)
        offsets = make_window_offsets(window=window, sparse_step=sparse_step)

        for g in groups:
            endpoints = valid_window_endpoints(len(g["X"]), window)
            expected = max(0, len(g["X"]) - window + 1)
            actual = len(endpoints)

            crosses_boundary = False
            bad_class = False
            non_monotonic = False

            if actual > 0:
                for end in endpoints[:min(actual, 10)]:
                    start = end - window + 1
                    idx_slice = g["index"][start:end + 1]
                    sampled_idx = g["index"][start + offsets]

                    traj_vals_full = pd.Index(idx_slice).get_level_values(0)
                    traj_vals_sampled = pd.Index(sampled_idx).get_level_values(0)
                    time_vals_full = pd.Index(idx_slice).get_level_values(-1)

                    if len(pd.unique(traj_vals_full)) != 1 or len(pd.unique(traj_vals_sampled)) != 1:
                        crosses_boundary = True
                    if pd.unique(g["y"]).shape[0] != 1:
                        bad_class = True
                    if not pd.Index(time_vals_full).is_monotonic_increasing:
                        non_monotonic = True

            rows.append({
                "trajectory_id": g["trajectory_id"],
                "class_value": g["class_value"],
                "group_len": len(g["X"]),
                "window": window,
                "sparse_step": sparse_step,
                "expected_windows": expected,
                "actual_windows": actual,
                "window_count_ok": expected == actual,
                "crosses_boundary": crosses_boundary,
                "single_class_ok": not bad_class,
                "time_monotonic_ok": not non_monotonic,
                "effective_window_points": len(offsets),
            })

    return pd.DataFrame(rows)

diag_train = validate_window_slicing(
    train_groups,
    windows=(1, 10, 100),
    sparse_step_map={1: None, 10: None, 100: 10},
)

display(diag_train.head(20))
print("all checks passed:",
      bool(diag_train["window_count_ok"].all()
           and (~diag_train["crosses_boundary"]).all()
           and diag_train["single_class_ok"].all()
           and diag_train["time_monotonic_ok"].all()))


,trajectory_id,class_value,group_len,window,sparse_step,expected_windows,actual_windows,window_count_ok,crosses_boundary,single_class_ok,time_monotonic_ok,effective_window_points
0,0,0,305220,1,NaN,305220,305220,True,False,True,True,1
1,1,1,305220,1,NaN,305220,305220,True,False,True,True,1
2,2,2,305220,1,NaN,305220,305220,True,False,True,True,1
3,3,3,305220,1,NaN,305220,305220,True,False,True,True,1
4,4,4,305220,1,NaN,305220,305220,True,False,True,True,1
5,5,5,305220,1,NaN,305220,305220,True,False,True,True,1
6,6,6,305220,1,NaN,305220,305220,True,False,True,True,1
7,7,7,305220,1,NaN,305220,305220,True,False,True,True,1
8,8,8,305220,1,NaN,305220,305220,True,False,True,True,1
9,9,9,305220,1,NaN,305220,305220,True,False,True,True,1


all checks passed: False


In [12]:

diag_summary = (
    diag_train
    .groupby(["window", "sparse_step"], dropna=False)
    .agg(
        n_trajectories=("trajectory_id", "count"),
        total_expected_windows=("expected_windows", "sum"),
        total_actual_windows=("actual_windows", "sum"),
        all_window_count_ok=("window_count_ok", "all"),
        any_crosses_boundary=("crosses_boundary", "any"),
        all_single_class_ok=("single_class_ok", "all"),
        all_time_monotonic_ok=("time_monotonic_ok", "all"),
    )
    .reset_index()
)
display(diag_summary)


,window,sparse_step,n_trajectories,total_expected_windows,total_actual_windows,all_window_count_ok,any_crosses_boundary,all_single_class_ok,all_time_monotonic_ok
0,1,NaN,15,4448700,4448700,True,False,True,True
1,10,NaN,15,4448565,4448565,True,True,True,True
2,100,10.0,15,4447215,4447215,True,True,True,True


In [13]:
RANDOM_STATE = 42
TRAIN_XGB_BATCH_SIZE = 50_000
XGB_ROUNDS_PER_BATCH = 25
VAL_BATCH_SIZE = 4_096
TEST_BATCH_SIZE = 4_096
ARTIFACTS_DIR = join("/home/ilya_treyvish/projects/lbnl_fdd/data", "artifacts_raw_windows")


In [14]:

def run_experiment_for_window(
    window: int,
    train_groups,
    val_groups,
    test_groups,
    random_state: int,
    val_batch_size: int = 4096,
    test_batch_size: int = 4096,
    sparse_step: int | None = None,
    version_name: str | None = None,
    train_xgb_batch_size: int = 50_000,
    xgb_rounds_per_batch: int = 25,
):
    if version_name is None:
        version_name = f"raw_window_{window}" if sparse_step is None else f"raw_window_{window}_step{sparse_step}"

    offsets = make_window_offsets(window=window, sparse_step=sparse_step)

    print(f"\n=== {version_name} | XGBoost(full train, batched, mixed) ===")
    batch_mix_preview = inspect_train_batch_mixing(
        train_groups,
        window=window,
        batch_size=train_xgb_batch_size,
        sparse_step=sparse_step,
        random_state=random_state,
        n_batches=5,
        idx_to_class=idx_to_class,
    )
    display(batch_mix_preview)

    t0 = time.perf_counter()
    xgb_model, xgb_train_info = fit_xgb_batched(
        train_groups=train_groups,
        window=window,
        classes_=classes_,
        batch_size=train_xgb_batch_size,
        sparse_step=sparse_step,
        random_state=random_state,
        rounds_per_batch=xgb_rounds_per_batch,
    )
    fit_seconds = time.perf_counter() - t0

    val_metrics, _, _ = evaluate_streaming(
        xgb_model,
        val_groups,
        window=window,
        classes_=classes_,
        batch_size=val_batch_size,
        sparse_step=sparse_step,
    )
    test_metrics, _, _ = evaluate_streaming(
        xgb_model,
        test_groups,
        window=window,
        classes_=classes_,
        batch_size=test_batch_size,
        sparse_step=sparse_step,
    )

    batch_stats_df = xgb_train_info["batch_class_stats"]
    train_features = len(offsets) * train_groups[0]["X"].shape[1]
    results_df = pd.DataFrame([{
        "version": version_name,
        "window": window,
        "sparse_step": -1 if sparse_step is None else sparse_step,
        "effective_window_points": len(offsets),
        "model_name": "XGBoostClassifier",
        "train_mode": "full_train_batched_mixed",
        "fit_seconds": fit_seconds,
        "train_samples": xgb_train_info["train_rows_seen"],
        "train_batches": xgb_train_info["train_batches"],
        "rounds_per_batch": xgb_train_info["rounds_per_batch"],
        "total_boost_rounds": xgb_train_info["total_boost_rounds"],
        "avg_batch_n_classes": float(batch_stats_df["n_classes_in_batch"].mean()),
        "avg_batch_max_class_share": float(batch_stats_df["max_class_share"].mean()),
        "train_features": train_features,
        "val_accuracy": val_metrics["accuracy"],
        "val_macro_f1": val_metrics["macro_f1"],
        "val_weighted_f1": val_metrics["weighted_f1"],
        "val_logloss": val_metrics["logloss"],
        "test_accuracy": test_metrics["accuracy"],
        "test_macro_f1": test_metrics["macro_f1"],
        "test_weighted_f1": test_metrics["weighted_f1"],
        "test_logloss": test_metrics["logloss"],
    }])

    artifacts = {
        "version": version_name,
        "window": window,
        "sparse_step": sparse_step,
        "offsets": offsets,
        "xgb_train_info": xgb_train_info,
        "batch_mix_preview": batch_mix_preview,
        "results_df": results_df,
        "models": {"XGBoostClassifier": xgb_model},
    }

    gc.collect()
    return artifacts



## Диагностика train-batch mixing

Перед запуском обучения для каждой конфигурации выше печатается `batch_mix_preview`:
- `n_classes_in_batch` — сколько классов попало в батч;
- `max_class_share` — доля самого частого класса в батче.

Для корректного batched XGBoost здесь не должно быть длинных одно-классовых батчей.


In [15]:
%%time
artifacts_raw_w1 = run_experiment_for_window(
    window=1,
    train_groups=train_groups,
    val_groups=val_groups,
    test_groups=test_groups,
    random_state=RANDOM_STATE,
    val_batch_size=VAL_BATCH_SIZE,
    test_batch_size=TEST_BATCH_SIZE,
    train_xgb_batch_size=TRAIN_XGB_BATCH_SIZE,
    xgb_rounds_per_batch=XGB_ROUNDS_PER_BATCH,
)
artifacts_raw_w1["results_df"]



=== raw_window_1 | XGBoost(full train, batched, mixed) ===


,batch_idx,batch_size,n_classes_in_batch,max_class_share,class_0,class_1,class_2,class_3,class_4,class_5,class_6,class_7,class_8,class_9,class_10,class_11,class_12,class_13,class_14
0,0,50000,15,0.06668,3334,3334,3333,3333,3333,3333,3333,3333,3334,3333,3333,3333,3334,3334,3333
1,1,50000,15,0.06668,3332,3333,3334,3334,3334,3334,3334,3334,3333,3334,3334,3333,3332,3332,3333
2,2,50000,15,0.06668,3334,3333,3333,3333,3333,3333,3333,3333,3333,3333,3333,3334,3334,3334,3334
3,3,50000,15,0.06668,3334,3333,3334,3333,3333,3333,3334,3333,3334,3333,3333,3334,3333,3333,3333
4,4,50000,15,0.06668,3333,3334,3333,3334,3334,3334,3332,3333,3332,3334,3333,3333,3334,3334,3333


CPU times: user 2h 3min 50s, sys: 18.7 s, total: 2h 4min 9s
Wall time: 19min 37s


,version,window,sparse_step,effective_window_points,model_name,train_mode,fit_seconds,train_samples,train_batches,rounds_per_batch,...,avg_batch_max_class_share,train_features,val_accuracy,val_macro_f1,val_weighted_f1,val_logloss,test_accuracy,test_macro_f1,test_weighted_f1,test_logloss
0,raw_window_1,1,-1,1,XGBoostClassifier,full_train_batched_mixed,570.329305,4448700,89,25,...,0.068624,25,0.697743,0.694688,0.694688,4.489151,0.515232,0.504388,0.52729,7.474155


In [ ]:
%%time
artifacts_raw_w10 = run_experiment_for_window(
    window=10,
    train_groups=train_groups,
    val_groups=val_groups,
    test_groups=test_groups,
    random_state=RANDOM_STATE,
    val_batch_size=VAL_BATCH_SIZE,
    test_batch_size=TEST_BATCH_SIZE,
    train_xgb_batch_size=TRAIN_XGB_BATCH_SIZE,
    xgb_rounds_per_batch=XGB_ROUNDS_PER_BATCH,
)
artifacts_raw_w10["results_df"]



=== raw_window_10 | XGBoost(full train, batched, mixed) ===


,batch_idx,batch_size,n_classes_in_batch,max_class_share,class_0,class_1,class_2,class_3,class_4,class_5,class_6,class_7,class_8,class_9,class_10,class_11,class_12,class_13,class_14
0,0,50000,15,0.06668,3334,3334,3333,3333,3333,3333,3333,3333,3334,3333,3333,3333,3334,3334,3333
1,1,50000,15,0.06668,3332,3333,3334,3334,3334,3334,3334,3334,3333,3334,3334,3333,3332,3332,3333
2,2,50000,15,0.06668,3334,3333,3333,3333,3333,3333,3333,3333,3333,3333,3333,3334,3334,3334,3334
3,3,50000,15,0.06668,3334,3333,3334,3333,3333,3333,3334,3333,3334,3333,3333,3334,3333,3333,3333
4,4,50000,15,0.06668,3333,3334,3333,3334,3334,3334,3332,3333,3332,3334,3333,3333,3334,3334,3333


In [ ]:
%%time
artifacts_raw_w100 = run_experiment_for_window(
    window=100,
    train_groups=train_groups,
    val_groups=val_groups,
    test_groups=test_groups,
    random_state=RANDOM_STATE,
    val_batch_size=VAL_BATCH_SIZE,
    test_batch_size=TEST_BATCH_SIZE,
    train_xgb_batch_size=TRAIN_XGB_BATCH_SIZE,
    xgb_rounds_per_batch=XGB_ROUNDS_PER_BATCH,
)
artifacts_raw_w100["results_df"]


Saved train window selection to: /home/ilya_treyvish/projects/lbnl_fdd/data/artifacts_raw_windows/raw_window_100_train_window_selection.jsonl

=== raw_window_100 | DummyClassifier ===

=== raw_window_100 | ExtraTreesClassifier ===

=== raw_window_100 | HistGradientBoostingClassifier ===
CPU times: user 1h 59min 37s, sys: 3min 16s, total: 2h 2min 54s
Wall time: 33min 4s


,version,window,sparse_step,effective_window_points,model_name,fit_seconds,train_samples,train_features,val_accuracy,val_macro_f1,val_weighted_f1,val_logloss,test_accuracy,test_macro_f1,test_weighted_f1,test_logloss
0,raw_window_100,100,-1,100,HistGradientBoostingClassifier,659.230621,50000,2500,0.863480,0.863574,0.863574,0.352309,0.812587,0.821828,0.813582,0.453709
1,raw_window_100,100,-1,100,ExtraTreesClassifier,44.136363,50000,2500,0.863170,0.862651,0.862651,0.406996,0.809150,0.816051,0.808569,0.477990
2,raw_window_100,100,-1,100,DummyClassifier,0.001266,50000,2500,0.066667,0.008333,0.008333,2.708050,0.069752,0.008694,0.009096,2.708046


In [ ]:
%%time
artifacts_raw_w100_sparse = run_experiment_for_window(
    window=100,
    train_groups=train_groups,
    val_groups=val_groups,
    test_groups=test_groups,
    random_state=RANDOM_STATE,
    sparse_step=10,
    val_batch_size=VAL_BATCH_SIZE,
    test_batch_size=TEST_BATCH_SIZE,
    train_xgb_batch_size=TRAIN_XGB_BATCH_SIZE,
    xgb_rounds_per_batch=XGB_ROUNDS_PER_BATCH,
)
artifacts_raw_w100_sparse["results_df"]


Saved train window selection to: /home/ilya_treyvish/projects/lbnl_fdd/data/artifacts_raw_windows/raw_window_100_sparse_step10_train_window_selection.jsonl

=== raw_window_100_sparse_step10 | DummyClassifier ===

=== raw_window_100_sparse_step10 | ExtraTreesClassifier ===

=== raw_window_100_sparse_step10 | HistGradientBoostingClassifier ===
CPU times: user 27min 6s, sys: 1min 17s, total: 28min 24s
Wall time: 8min 53s


,version,window,sparse_step,effective_window_points,model_name,fit_seconds,train_samples,train_features,val_accuracy,val_macro_f1,val_weighted_f1,val_logloss,test_accuracy,test_macro_f1,test_weighted_f1,test_logloss
0,raw_window_100_sparse_step10,100,10,10,HistGradientBoostingClassifier,59.178224,50000,250,0.862401,0.862614,0.862614,0.361282,0.809392,0.818886,0.810567,0.475031
1,raw_window_100_sparse_step10,100,10,10,ExtraTreesClassifier,10.310575,50000,250,0.859335,0.858801,0.858801,0.378840,0.803024,0.809935,0.802255,0.500158
2,raw_window_100_sparse_step10,100,10,10,DummyClassifier,0.002522,50000,250,0.066667,0.008333,0.008333,2.708050,0.069752,0.008694,0.009096,2.708046


In [ ]:

comparison = pd.concat(
    [
        artifacts_raw_w1["results_df"],
        artifacts_raw_w10["results_df"],
        artifacts_raw_w100["results_df"],
        artifacts_raw_w100_sparse["results_df"],
    ],
    axis=0,
).reset_index(drop=True)

comparison = comparison[
    [
        "version",
        "window",
        "sparse_step",
        "effective_window_points",
        "model_name",
        "train_mode",
        "fit_seconds",
        "train_samples",
        "train_batches",
        "rounds_per_batch",
        "total_boost_rounds",
        "avg_batch_n_classes",
        "avg_batch_max_class_share",
        "train_features",
        "val_accuracy",
        "val_macro_f1",
        "val_weighted_f1",
        "val_logloss",
        "test_accuracy",
        "test_macro_f1",
        "test_weighted_f1",
        "test_logloss",
    ]
]
comparison


,version,window,sparse_step,effective_window_points,model_name,fit_seconds,train_samples,train_features,val_accuracy,val_macro_f1,val_weighted_f1,val_logloss,test_accuracy,test_macro_f1,test_weighted_f1,test_logloss
0,raw_window_100,100,-1,100,HistGradientBoostingClassifier,659.230621,50000,2500,0.863480,0.863574,0.863574,0.352309,0.812587,0.821828,0.813582,0.453709
1,raw_window_100,100,-1,100,ExtraTreesClassifier,44.136363,50000,2500,0.863170,0.862651,0.862651,0.406996,0.809150,0.816051,0.808569,0.477990
2,raw_window_100_sparse_step10,100,10,10,HistGradientBoostingClassifier,59.178224,50000,250,0.862401,0.862614,0.862614,0.361282,0.809392,0.818886,0.810567,0.475031
3,raw_window_100_sparse_step10,100,10,10,ExtraTreesClassifier,10.310575,50000,250,0.859335,0.858801,0.858801,0.378840,0.803024,0.809935,0.802255,0.500158
4,raw_window_10,10,-1,10,ExtraTreesClassifier,11.935301,50000,250,0.850026,0.849486,0.849486,0.394850,0.788101,0.794683,0.787384,0.506769
5,raw_window_1,1,-1,1,ExtraTreesClassifier,2.021577,50000,25,0.846483,0.845614,0.845614,0.448223,0.780146,0.787443,0.779940,0.602758
6,raw_window_10,10,-1,10,HistGradientBoostingClassifier,81.428125,50000,250,0.843416,0.843395,0.843395,0.400401,0.785601,0.795642,0.786334,0.520352
7,raw_window_1,1,-1,1,HistGradientBoostingClassifier,8.942326,50000,25,0.839617,0.839765,0.839765,0.421813,0.783813,0.792421,0.783958,0.545815
8,raw_window_1,1,-1,1,DummyClassifier,0.018611,50000,25,0.066667,0.008333,0.008333,2.708050,0.069750,0.008694,0.009096,2.708046
9,raw_window_10,10,-1,10,DummyClassifier,0.001218,50000,250,0.066667,0.008333,0.008333,2.708050,0.069750,0.008694,0.009096,2.708046


## Detailed report по XGBoost для каждой конфигурации

In [ ]:
experiment_specs = [
    ("raw_window_1", artifacts_raw_w1, 1, None),
    ("raw_window_10", artifacts_raw_w10, 10, None),
    ("raw_window_100", artifacts_raw_w100, 100, None),
    ("raw_window_100_sparse_step10", artifacts_raw_w100_sparse, 100, 10),
]

for version_name, artifacts, window, sparse_step in experiment_specs:
    model = artifacts["models"]["XGBoostClassifier"]

    _, y_true_val, y_pred_val = evaluate_streaming(
        model,
        val_groups,
        window=window,
        classes_=classes_,
        batch_size=VAL_BATCH_SIZE,
        sparse_step=sparse_step,
    )

    print("=" * 100)
    print(f"Model for {version_name}: XGBoostClassifier")
    print(classification_report(y_true_val, y_pred_val, zero_division=0))


Best model for raw_window_1: ExtraTreesClassifier
              precision    recall  f1-score   support

           0       0.53      0.54      0.54     87840
           1       0.85      0.78      0.81     87840
           2       0.89      0.96      0.93     87840
           3       0.75      0.71      0.73     87840
           4       0.74      0.84      0.79     87840
           5       0.71      0.62      0.66     87840
           6       0.98      0.98      0.98     87840
           7       1.00      0.99      0.99     87840
           8       1.00      1.00      1.00     87840
           9       1.00      1.00      1.00     87840
          10       0.67      0.72      0.69     87840
          11       1.00      1.00      1.00     87840
          12       1.00      1.00      1.00     87840
          13       0.99      0.99      0.99     87840
          14       0.60      0.57      0.58     87840

    accuracy                           0.85   1317600
   macro avg       0.85      0